# CNN Image Classification — FashionMNIST + Real-World Smartphone Photos
**Student ID:** `REPLACE_WITH_YOUR_ID`  
**Dataset:** FashionMNIST (Option 1)  
**Custom classes photographed:** T-shirt/top, Trouser, Sneaker, Bag (10 photos total, mix of these 4 classes)

This notebook is fully automated: clicking **Run All** will
1. Clone this repo (pulls your `dataset/` phone photos and, if present, a saved `model/*.pth`),
2. Download FashionMNIST via `torchvision`,
3. Build and train a CNN from scratch (or load saved weights),
4. Evaluate on the standard test set (confusion matrix + error analysis),
5. Preprocess and classify your 10 phone photos, printing predicted class + confidence for each.

> **Before you submit:** replace `GITHUB_REPO_URL` below with your own repo URL, and make sure your
> real smartphone photos are committed to `dataset/` in that repo (see the fallback-image note in Cell 4).

## 0. Configuration

In [ ]:

GITHUB_REPO_URL = "https://github.com/chowdhurymohammadtushar-sudo/CNN.git"  
STUDENT_ID = "220102" 

REPO_DIR = "repo"
IMG_SIZE = 28              
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 1e-3
VAL_SPLIT = 0.1            
SEED = 42

CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

## 1. Imports & Setup

In [ ]:
import os, glob, random, shutil
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision import datasets, transforms

from PIL import Image
from sklearn.metrics import confusion_matrix
import seaborn as sns

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Automated Data Retrieval (GitHub clone)
No manual uploads. This clones your repo so `dataset/` (your 10 phone photos) and, if you've
already trained once, `model/{STUDENT_ID}.pth` are available on disk.

In [ ]:
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

clone_ok = os.system(f"git clone {GITHUB_REPO_URL} {REPO_DIR}") == 0

CUSTOM_DIR = os.path.join(REPO_DIR, "dataset")
MODEL_DIR = os.path.join(REPO_DIR, "model")
os.makedirs(CUSTOM_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

existing_photos = [f for f in glob.glob(os.path.join(CUSTOM_DIR, "*"))
                    if f.lower().endswith((".jpg", ".jpeg", ".png"))]
print(f"Clone succeeded: {clone_ok}")
print(f"Found {len(existing_photos)} custom images in {CUSTOM_DIR}")

### Fallback: synthetic placeholder photos
If the clone fails or `dataset/` is empty (e.g. you're test-driving this notebook before your
repo has real photos), we generate simple placeholder images so the whole pipeline still runs
end-to-end. **Replace these with your actual smartphone photos before submitting** — the
filenames just need to start with the class name (e.g. `sneaker_1.jpg`, `bag_2.jpg`, `tshirt_3.jpg`,
`trouser_4.jpg`) so the labeling step below can match them automatically.

In [ ]:
PLACEHOLDER_CLASSES = ["tshirt", "trouser", "sneaker", "bag"]

def make_placeholder_photos(out_dir, n=10):
    from PIL import ImageDraw
    for i in range(n):
        cls = PLACEHOLDER_CLASSES[i % len(PLACEHOLDER_CLASSES)]
        img = Image.new("RGB", (400, 400), (245, 245, 245))
        draw = ImageDraw.Draw(img)
        # crude shape per class, just so the file isn't blank
        if cls == "tshirt":
            draw.polygon([(100,120),(160,80),(240,80),(300,120),(260,160),
                          (260,320),(140,320),(140,160)], fill=(80,120,200))
        elif cls == "trouser":
            draw.polygon([(150,80),(250,80),(260,320),(210,320),(200,200),
                          (190,320),(140,320)], fill=(60,60,60))
        elif cls == "sneaker":
            draw.polygon([(80,260),(320,260),(320,300),(80,300)], fill=(200,60,60))
            draw.polygon([(80,200),(260,200),(300,260),(80,260)], fill=(220,90,90))
        elif cls == "bag":
            draw.rectangle([(120,150),(280,320)], fill=(150,100,60))
            draw.arc([(140,90),(260,190)], 180, 360, fill=(150,100,60), width=8)
        path = os.path.join(out_dir, f"{cls}_{i+1}.jpg")
        img.save(path)
    print(f"Generated {n} placeholder photos in {out_dir}")

existing_photos = [f for f in glob.glob(os.path.join(CUSTOM_DIR, "*"))
                    if f.lower().endswith((".jpg", ".jpeg", ".png"))]
if len(existing_photos) == 0:
    print("No custom photos found — generating placeholders for a full dry run.")
    make_placeholder_photos(CUSTOM_DIR, n=10)
    existing_photos = [f for f in glob.glob(os.path.join(CUSTOM_DIR, "*"))
                        if f.lower().endswith((".jpg", ".jpeg", ".png"))]

print("Custom images ready:", existing_photos)

## 3. Standard Dataset: FashionMNIST

In [ ]:
# Grayscale, 28x28 -> ToTensor -> Normalize (FashionMNIST mean/std)
MEAN, STD = (0.2860,), (0.3530,)

# Training gets light augmentation (rotation/shift/zoom) so the model generalizes better
# to real smartphone photos, which are never perfectly centered or perfectly upright.
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
# Val/Test stay clean (no augmentation) so evaluation reflects real accuracy
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Two views of the same training data: one augmented (for train), one clean (for val)
full_train_aug   = datasets.FashionMNIST(root="data", train=True, download=True, transform=train_transform)
full_train_clean = datasets.FashionMNIST(root="data", train=True, download=True, transform=eval_transform)
test_set         = datasets.FashionMNIST(root="data", train=False, download=True, transform=eval_transform)

indices = list(range(len(full_train_aug)))
random.Random(SEED).shuffle(indices)
val_size = int(len(indices) * VAL_SPLIT)
val_indices, train_indices = indices[:val_size], indices[val_size:]

from torch.utils.data import Subset
train_set = Subset(full_train_aug, train_indices)     # augmented
val_set   = Subset(full_train_clean, val_indices)     # clean

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

### 3.1 Exploratory Data Analysis
Before building the model, let's look at what we're working with: sample images and how
balanced the classes are.

In [ ]:
# Sample grid: one example per class
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()
shown = set()
for img, label in full_train_clean:
    if label not in shown:
        axes[label].imshow(img.squeeze().numpy() * STD[0] + MEAN[0], cmap="gray")
        axes[label].set_title(CLASS_NAMES[label])
        axes[label].axis("off")
        shown.add(label)
    if len(shown) == 10:
        break
plt.suptitle("One Example per Class — FashionMNIST")
plt.tight_layout()
plt.show()

In [ ]:
# Class balance check
train_labels = [full_train_clean[i][1] for i in range(len(full_train_clean))]
counts = np.bincount(train_labels, minlength=10)

plt.figure(figsize=(9, 4))
plt.bar(CLASS_NAMES, counts, color="steelblue")
plt.title("Training Set — Images per Class")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
print("Class counts:", dict(zip(CLASS_NAMES, counts.tolist())))

## 4. CNN Model Architecture

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout(0.25)
        # 28 -> pool -> 14 -> pool -> 7
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # 28x28 -> 14x14
        x = self.pool(self.relu(self.conv2(x)))   # 14x14 -> 7x7
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = CNN(num_classes=10).to(device)
print(model)

### 4.1 Model Summary

In [ ]:
def count_parameters(m):
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

total_params, trainable_params = count_parameters(model)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Layer-by-layer shape trace using a dummy input
dummy = torch.zeros(1, 1, IMG_SIZE, IMG_SIZE).to(device)
x = dummy
with torch.no_grad():
    x = model.pool(model.relu(model.conv1(x))); print("After conv1+pool:", tuple(x.shape))
    x = model.pool(model.relu(model.conv2(x))); print("After conv2+pool:", tuple(x.shape))
    x = x.view(x.size(0), -1);                   print("After flatten:   ", tuple(x.shape))
    x = model.relu(model.fc1(x));                 print("After fc1:       ", tuple(x.shape))
    x = model.fc2(x);                             print("After fc2:       ", tuple(x.shape))

## 5. Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

def run_epoch(loader, train_mode):
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train_mode):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if train_mode:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train_mode:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

In [ ]:
MODEL_PATH = os.path.join(MODEL_DIR, f"{STUDENT_ID}.pth")
LOCAL_MODEL_PATH = f"{STUDENT_ID}.pth"

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

if os.path.isfile(MODEL_PATH):
    print(f"Found saved weights at {MODEL_PATH} — loading instead of retraining.")
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
else:
    print("No saved weights found — training from scratch.")
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = run_epoch(train_loader, train_mode=True)
        va_loss, va_acc = run_epoch(val_loader, train_mode=False)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)
        print(f"Epoch {epoch:2d}/{EPOCHS} | "
              f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
              f"val_loss={va_loss:.4f} val_acc={va_acc:.4f}")

    # Save state dict locally (upload this file to your repo's model/ folder)
    torch.save(model.state_dict(), LOCAL_MODEL_PATH)
    print(f"Saved trained weights to ./{LOCAL_MODEL_PATH}")
    print("--> Commit this file to your GitHub repo under model/ so future 'Run All' loads it directly.")

## 6. Training History Plots

In [ ]:
if history["train_loss"]:
    epochs_range = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs_range, history["train_loss"], label="Train Loss")
    axes[0].plot(epochs_range, history["val_loss"], label="Val Loss")
    axes[0].set_title("Loss vs Epochs")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()

    axes[1].plot(epochs_range, history["train_acc"], label="Train Acc")
    axes[1].plot(epochs_range, history["val_acc"], label="Val Acc")
    axes[1].set_title("Accuracy vs Epochs")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print("Loaded pre-trained weights, so there's no fresh training history to plot this run.")

## 7. Confusion Matrix (Standard Test Set)

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

test_acc = (np.array(all_preds) == np.array(all_labels)).mean()
print(f"Test set accuracy: {test_acc:.4f}")

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix — FashionMNIST Test Set")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 7.1 Classification Report & Per-Class Accuracy

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=3)
print(report)

In [ ]:
# Per-class accuracy bar chart (diagonal of confusion matrix / row sum)
per_class_acc = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(9, 4))
plt.bar(CLASS_NAMES, per_class_acc, color="seagreen")
plt.axhline(test_acc, color="red", linestyle="--", label=f"Overall acc = {test_acc:.3f}")
plt.title("Per-Class Accuracy — Test Set")
plt.ylabel("Accuracy"); plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Visual Error Analysis (3 Misclassified Test Images)

In [ ]:
all_preds_arr = np.array(all_preds)
all_labels_arr = np.array(all_labels)
wrong_idx = np.where(all_preds_arr != all_labels_arr)[0]
sample_wrong = np.random.choice(wrong_idx, size=min(3, len(wrong_idx)), replace=False)

fig, axes = plt.subplots(1, len(sample_wrong), figsize=(4 * len(sample_wrong), 4))
if len(sample_wrong) == 1:
    axes = [axes]
for ax, idx in zip(axes, sample_wrong):
    img_tensor, true_label = test_set[idx]
    img = img_tensor.squeeze().numpy() * STD[0] + MEAN[0]  # unnormalize for display
    ax.imshow(img, cmap="gray")
    ax.set_title(f"True: {CLASS_NAMES[true_label]}\nPred: {CLASS_NAMES[all_preds_arr[idx]]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Real-World Prediction — Your Smartphone Photos
**Why naive preprocessing fails on real photos:** FashionMNIST images are tiny, perfectly
centered garments on a **plain dark background** — almost the opposite of a phone photo, which
has a **light background**, the item off-center, and lots of empty space around it. Simply
resizing a phone photo to 28x28 and normalizing it looks nothing like what the model was
trained on, so predictions come out close to random.

To close that gap, `preprocess_custom_image` below does what MNIST-style digit-recognizer apps
do: it (1) **auto-inverts** the image if the background is light (so the background becomes dark,
matching FashionMNIST), (2) **finds and crops to the object** using a brightness threshold,
discarding empty background, and (3) **resizes and centers** the object on a 28x28 canvas the
same way MNIST/FashionMNIST images are formatted. This is the single biggest lever for real-world
accuracy — even a well-trained model cannot succeed if its input doesn't resemble its training data.

In [ ]:
def preprocess_custom_image(path, debug=False):
    img = Image.open(path).convert("L")  # grayscale, matches FashionMNIST
    arr = np.array(img).astype(np.float32)

    # 1) Auto-invert: FashionMNIST = light garment on DARK background.
    #    A phone photo on a table is usually the opposite (dark item on a light surface).
    border_px = np.concatenate([arr[0, :], arr[-1, :], arr[:, 0], arr[:, -1]])
    if border_px.mean() > 127:  # light background detected -> invert
        arr = 255.0 - arr

    # 2) Threshold + crop to the object's bounding box, discarding empty background.
    thresh = arr.mean() + 0.4 * arr.std()
    mask = arr > thresh
    ys, xs = np.where(mask)
    if len(ys) > 20 and len(xs) > 20:  # only crop if we found a plausible object
        pad = int(0.06 * max(arr.shape))
        y0, y1 = max(ys.min() - pad, 0), min(ys.max() + pad, arr.shape[0])
        x0, x1 = max(xs.min() - pad, 0), min(xs.max() + pad, arr.shape[1])
        arr = arr[y0:y1, x0:x1]

    cropped = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

    # 3) Resize preserving aspect ratio into a 20x20 box, then paste centered on a 28x28
    #    dark canvas — this mirrors exactly how MNIST-family datasets are formatted.
    w, h = cropped.size
    scale = 20.0 / max(w, h)
    new_w, new_h = max(1, round(w * scale)), max(1, round(h * scale))
    resized = cropped.resize((new_w, new_h), Image.LANCZOS)

    canvas = Image.new("L", (IMG_SIZE, IMG_SIZE), 0)  # black background
    offset = ((IMG_SIZE - new_w) // 2, (IMG_SIZE - new_h) // 2)
    canvas.paste(resized, offset)

    tensor = transforms.ToTensor()(canvas)
    tensor = transforms.Normalize(MEAN, STD)(tensor)
    return canvas, tensor.unsqueeze(0)

custom_paths = sorted(glob.glob(os.path.join(CUSTOM_DIR, "*")))
custom_paths = [p for p in custom_paths if p.lower().endswith((".jpg", ".jpeg", ".png"))]
print(f"Classifying {len(custom_paths)} custom photos...")

results = []
model.eval()
with torch.no_grad():
    for path in custom_paths:
        raw_img, tensor = preprocess_custom_image(path)
        tensor = tensor.to(device)
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = int(np.argmax(probs))
        confidence = probs[pred_idx] * 100
        results.append((path, raw_img, CLASS_NAMES[pred_idx], confidence))
        print(f"{os.path.basename(path):20s} -> Pred: {CLASS_NAMES[pred_idx]} ({confidence:.1f}%)")

### 9.0 Debug View: Original Photo vs. What the Model Actually Sees
If predictions still look wrong, check this grid first. If the 28x28 version doesn't clearly
look like the object (badly cropped, inverted wrong, or too much background left in), that's
the fix to make — not the model architecture. Common causes: cluttered/patterned background,
item too small in the frame, or shadows confusing the brightness threshold. Retake the photo
with the item large in frame on a single plain surface (a white sheet of paper or plain floor)
for the most reliable result.

In [ ]:
n = len(custom_paths)
fig, axes = plt.subplots(2, n, figsize=(2.4 * n, 5))
for i, path in enumerate(custom_paths):
    original = Image.open(path)
    processed, _ = preprocess_custom_image(path)
    axes[0, i].imshow(original)
    axes[0, i].set_title(os.path.basename(path), fontsize=8)
    axes[0, i].axis("off")
    axes[1, i].imshow(processed, cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original", fontsize=10)
axes[1, 0].set_ylabel("Model input (28x28)", fontsize=10)
plt.suptitle("Top row: original photo   |   Bottom row: preprocessed model input")
plt.tight_layout()
plt.show()

### Custom Prediction Gallery

In [ ]:
n = len(results)
cols = 5
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3.3 * rows))
axes = np.array(axes).reshape(-1)

for ax, (path, raw_img, pred_class, conf) in zip(axes, results):
    ax.imshow(raw_img, cmap="gray")
    ax.set_title(f"Pred: {pred_class} ({conf:.1f}%)", fontsize=10)
    ax.axis("off")

for ax in axes[len(results):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

### 9.1 Top-3 Predictions per Photo
Sometimes the top-1 class is wrong but the correct class is a close second — useful to inspect
when debugging which real-world items confuse the model.

In [ ]:
model.eval()
with torch.no_grad():
    for path in custom_paths:
        _, tensor = preprocess_custom_image(path)
        tensor = tensor.to(device)
        probs = torch.softmax(model(tensor), dim=1).cpu().numpy()[0]
        top3_idx = probs.argsort()[-3:][::-1]
        top3_str = ", ".join(f"{CLASS_NAMES[i]} ({probs[i]*100:.1f}%)" for i in top3_idx)
        print(f"{os.path.basename(path):20s} -> {top3_str}")

### 9.2 Custom Photo Results Table

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "filename": [os.path.basename(p) for p, _, _, _ in results],
    "predicted_class": [c for _, _, c, _ in results],
    "confidence_%": [round(cf, 1) for _, _, _, cf in results],
})
results_df